In [ ]:
# English -> Tamil Neural Machine Translation
# Encoder-Decoder + Bahdanau Attention

import os, re, random, urllib.request
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

# -----------------------------
# 1. Download English-Tamil Dataset
# -----------------------------

EN_URL = "https://raw.githubusercontent.com/nlpc-uom/English-Tamil-Parallel-Corpus/master/En-Ta%20Corpus/En-Ta%20English.txt"
TA_URL = "https://raw.githubusercontent.com/nlpc-uom/English-Tamil-Parallel-Corpus/master/En-Ta%20Corpus/En-Ta%20Tamil.txt"

if not os.path.exists("english.txt"):
    urllib.request.urlretrieve(EN_URL, "english.txt")

if not os.path.exists("tamil.txt"):
    urllib.request.urlretrieve(TA_URL, "tamil.txt")

# -----------------------------
# 2. Load Dataset
# -----------------------------

with open("english.txt", encoding="utf-8-sig") as f:
    en = f.readlines()[3:]

with open("tamil.txt", encoding="utf-8-sig") as f:
    ta = f.readlines()[3:]

pairs = [(e.strip(), t.strip()) for e, t in zip(en, ta)
         if e.strip() and t.strip()]

random.seed(42)
random.shuffle(pairs)

# Keep a manageable dataset for CPU training
pairs = pairs[:6000]

train_pairs = pairs[:5000]
test_pairs = pairs[5000:]

print("Training pairs:", len(train_pairs))
print("Testing pairs :", len(test_pairs))

# -----------------------------
# 3. Tokenization
# -----------------------------

def tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text.lower(), re.UNICODE)

# Special tokens
PAD, UNK, START, END = 0, 1, 2, 3

def build_vocab(sentences):
    vocab = {"<PAD>": PAD, "<UNK>": UNK,
             "<START>": START, "<END>": END}

    for sentence in sentences:
        for word in tokenize(sentence):
            if word not in vocab:
                vocab[word] = len(vocab)

    return vocab

en_vocab = build_vocab([x[0] for x in train_pairs])
ta_vocab = build_vocab([x[1] for x in train_pairs])

en_id = {v:k for k,v in en_vocab.items()}
ta_id = {v:k for k,v in ta_vocab.items()}

print("English vocabulary:", len(en_vocab))
print("Tamil vocabulary  :", len(ta_vocab))

MAX_LEN = 20

def encode(sentence, vocab):
    words = tokenize(sentence)[:MAX_LEN-2]
    ids = [START] + [vocab.get(w, UNK) for w in words] + [END]
    ids += [PAD] * (MAX_LEN - len(ids))
    return ids[:MAX_LEN]

# -----------------------------
# 4. Dataset
# -----------------------------

class TranslationDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        e, t = self.pairs[i]
        return (
            torch.tensor(encode(e, en_vocab)),
            torch.tensor(encode(t, ta_vocab))
        )

train_loader = DataLoader(
    TranslationDataset(train_pairs),
    batch_size=32,
    shuffle=True
)

# -----------------------------
# 5. Encoder
# -----------------------------

class Encoder(nn.Module):
    def __init__(self, vocab_size, emb=128, hidden=128):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, emb, padding_idx=PAD)

        self.lstm = nn.LSTM(
            emb, hidden,
            batch_first=True
        )

    def forward(self, x):
        x = self.embedding(x)
        outputs, (hidden, cell) = self.lstm(x)
        return outputs, hidden, cell

# -----------------------------
# 6. Bahdanau Attention
# -----------------------------

class Attention(nn.Module):
    def __init__(self, hidden):
        super().__init__()

        self.attn = nn.Linear(hidden * 2, hidden)
        self.score = nn.Linear(hidden, 1)

    def forward(self, hidden, encoder_outputs):

        # hidden: [1,B,H]
        hidden = hidden[-1].unsqueeze(1)

        # [B,S,H]
        energy = torch.tanh(
            self.attn(
                torch.cat(
                    (encoder_outputs, hidden.expand_as(encoder_outputs)),
                    dim=2
                )
            )
        )

        scores = self.score(energy).squeeze(2)

        weights = torch.softmax(scores, dim=1)

        context = torch.bmm(
            weights.unsqueeze(1),
            encoder_outputs
        )

        return context, weights

# -----------------------------
# 7. Decoder
# -----------------------------

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb=128, hidden=128):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size, emb, padding_idx=PAD
        )

        self.attention = Attention(hidden)

        self.lstm = nn.LSTM(
            emb + hidden,
            hidden,
            batch_first=True
        )

        self.fc = nn.Linear(hidden, vocab_size)

    def forward(self, x, hidden, cell, encoder_outputs):

        x = self.embedding(x.unsqueeze(1))

        context, weights = self.attention(
            hidden, encoder_outputs
        )

        x = torch.cat((x, context), dim=2)

        output, (hidden, cell) = self.lstm(
            x, (hidden, cell)
        )

        prediction = self.fc(output.squeeze(1))

        return prediction, hidden, cell, weights

# -----------------------------
# 8. Seq2Seq Model
# -----------------------------

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg, teacher_forcing=0.5):

        batch = src.size(0)

        outputs = torch.zeros(
            batch,
            MAX_LEN,
            len(ta_vocab)
        )

        encoder_outputs, hidden, cell = self.encoder(src)

        input = trg[:, 0]

        for t in range(1, MAX_LEN):

            output, hidden, cell, _ = self.decoder(
                input,
                hidden,
                cell,
                encoder_outputs
            )

            outputs[:, t] = output

            best = output.argmax(1)

            input = (
                trg[:, t]
                if random.random() < teacher_forcing
                else best
            )

        return outputs

# -----------------------------
# 9. Create Model
# -----------------------------

device = torch.device("cpu")

encoder = Encoder(len(en_vocab))
decoder = Decoder(len(ta_vocab))

model = Seq2Seq(encoder, decoder).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

criterion = nn.CrossEntropyLoss(
    ignore_index=PAD
)

# -----------------------------
# 10. Training
# -----------------------------

EPOCHS = 5

for epoch in range(EPOCHS):

    model.train()
    total_loss = 0

    for src, trg in train_loader:

        src = src.to(device)
        trg = trg.to(device)

        optimizer.zero_grad()

        output = model(src, trg)

        loss = criterion(
            output[:, 1:].reshape(-1, len(ta_vocab)),
            trg[:, 1:].reshape(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(), 1
        )

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{EPOCHS} "
        f"Loss: {total_loss/len(train_loader):.4f}"
    )

# -----------------------------
# 11. Translation Function
# -----------------------------

def translate(sentence):

    model.eval()

    src = torch.tensor(
        [encode(sentence, en_vocab)]
    ).to(device)

    with torch.no_grad():

        encoder_outputs, hidden, cell = model.encoder(src)

        input = torch.tensor([START]).to(device)

        result = []
        attention_list = []

        for _ in range(MAX_LEN):

            output, hidden, cell, weights = model.decoder(
                input,
                hidden,
                cell,
                encoder_outputs
            )

            next_word = output.argmax(1).item()

            attention_list.append(
                weights.squeeze().numpy()
            )

            if next_word == END:
                break

            if next_word not in [PAD, START]:
                result.append(ta_id.get(next_word, "<UNK>"))

            input = torch.tensor([next_word]).to(device)

    return " ".join(result), attention_list

# -----------------------------
# 12. Test on Unseen Sentence
# -----------------------------

sentence = test_pairs[0][0]

translation, attention = translate(sentence)

print("\nUnseen English:")
print(sentence)

print("\nGenerated Tamil:")
print(translation)

print("\nActual Tamil:")
print(test_pairs[0][1])

# -----------------------------
# 13. Required Example
# -----------------------------

demo = "How are you?"

translation, attention = translate(demo)

print("\nDemo Input:")
print("How are you?")

print("\nGenerated Tamil:")
print(translation)

# -----------------------------
# 14. Attention Visualization
# -----------------------------

words = tokenize(demo)

if len(attention) > 0:

    A = torch.tensor(attention).numpy()

    A = A[:, :len(words)]

    plt.figure(figsize=(9, 5))

    plt.imshow(A, aspect="auto")

    plt.xticks(
        range(len(words)),
        words,
        rotation=45
    )

    plt.xlabel("English Source Words")
    plt.ylabel("Generated Tamil Words")

    plt.title("Attention over English Source Words")

    plt.colorbar(label="Attention Weight")

    plt.tight_layout()
    plt.show()

Training pairs: 5000
Testing pairs : 1000
English vocabulary: 4970
Tamil vocabulary  : 1526
Epoch 1/5 Loss: 4.1717
Epoch 2/5 Loss: 3.4078
Epoch 3/5 Loss: 3.2189
Epoch 4/5 Loss: 3.0947
Epoch 5/5 Loss: 2.9979

Unseen English:
4. Real time video streaming of Rupavahini channel throughout the transmission.

Generated Tamil:
1 . 1 . ் ் த ் த ் த ் த ் த ் த ்

Actual Tamil:
4.	ரூபவாஹினி கோபுர நேரத்திலேயே அதனை இணையத்தளம் ஊடாக ஒளிபரப்புவதற்கு நடவடிக்கை மேற்கொள்ளல்

Demo Input:
How are you?

Generated Tamil:
ப ் த ி த ் த ் த ் த ் த ் த ் ப ்


C:\Users\lenovo\AppData\Local\Temp\ipykernel_31008\3345152793.py:397: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:255.)
  A = torch.tensor(attention).numpy()
